In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "source_system")
dbutils.widgets.text("table_name", "", "table_name")
dbutils.widgets.text("landing_timestamp", "", "landing_timestamp")
dbutils.widgets.text("source_file_format", "", "source_file_format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Build the landing path once and reuse it
landing_path = f"/Volumes/adb_caresync/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/"

def path_has_files(path):
    """Return True if the path exists and contains at least one entry.
    dbutils.fs.ls raises an exception if the path or any parent directory is missing."""
    try:
        return len(dbutils.fs.ls(path)) > 0
    except Exception:
        return False

if not path_has_files(landing_path):
    msg = f"No new files received for {source_system}/{table_name} at {landing_timestamp}. Path not found or empty: {landing_path}"
    print(msg)
    # Exit gracefully (notebook run succeeds, no failure)
    dbutils.notebook.exit(msg)

print(f"Landing path found: {landing_path}")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")